
# IMTalker - Generator Training: Audio Preprocessing (notebook 2/3)

Extracts **wav2vec2 audio features** for Generator Training (README: *Train -> 2. Train the generator ->
Data Preparation*): *"final-layer features extracted from audio WAV files using Wav2Vec2"*.

For every cropped video from `01_preprocess_video.ipynb` (`videos_512/<stem>.mp4`) with a matching
`<stem>.wav` in your audio folder, this notebook produces:

    generator_dataset/audio/<stem>.npy    # float32 ndarray of shape (T, 768)

where `T` is the number of frames of the cropped video and the features are the `last_hidden_state` of
`facebook/wav2vec2-base-960h`, **resampled to the video frame rate (25 fps)** exactly like
`generator/FM.py: AudioEncoder` does at training / inference time:

1. `librosa.load(wav, sr=16000)` + `Wav2Vec2FeatureExtractor` - identical to `app.py: process_audio` /
   `generator/generate.py: default_aud_loader`
2. pad (replicate) / trim the waveform to `T * 16000 / 25` samples - identical to `FM.py: AudioEncoder._pad_audio`
3. `Wav2VecModel` (the repo's own `generator/wav2vec2.py`) with `seq_len=T` - the model's
   `linear_interpolation` resamples the 49 Hz wav2vec2 grid onto the T video frames
4. save `last_hidden_state` (`only_last_features=True`, `generator/options/base_options.py`)

Run this **after** `01_preprocess_video.ipynb` and before `03_verify_dataset.ipynb`.


In [ ]:

# ============================================================================
# CONFIGURATION - EDIT THE PATHS BELOW
# ============================================================================
from pathlib import Path
import torch

# --- Input: folder with your .wav files --------------------------------------
RAW_AUDIO_DIR = Path(r"/workspace/IMTalker/assets/audio")   # <-- CHANGE ME

# --- Repository root ----------------------------------------------------------
REPO_ROOT = Path(r"/workspace/IMTalker")

# --- Paths produced by 01_preprocess_video.ipynb ------------------------------
WORK_DIR = REPO_ROOT / "preprocessing_output"
CROPPED_VIDEO_DIR = WORK_DIR / "videos_512"
DATASET_DIR = WORK_DIR / "generator_dataset"
AUDIO_DIR = DATASET_DIR / "audio"
CKPT_DIR = WORK_DIR / "checkpoints"
WAV2VEC_DIR = CKPT_DIR / "wav2vec2-base-960h"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OVERWRITE = False

assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"
assert RAW_AUDIO_DIR.exists(), f"RAW_AUDIO_DIR not found: {RAW_AUDIO_DIR} - edit the config above!"
assert CROPPED_VIDEO_DIR.exists(), "Run 01_preprocess_video.ipynb first (no videos_512 folder)"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

cropped_videos = sorted(CROPPED_VIDEO_DIR.glob("*.mp4"))
wavs = {p.stem: p for p in RAW_AUDIO_DIR.iterdir() if p.suffix.lower() == ".wav"}
print(f"{len(cropped_videos)} cropped videos | {len(wavs)} wav files")

missing_wav = [v.stem for v in cropped_videos if v.stem not in wavs]
unused_wav = [s for s in wavs if s not in {v.stem for v in cropped_videos}]
if missing_wav:
    print(f"[warn] {len(missing_wav)} videos without a matching .wav:", missing_wav[:5])
if unused_wav:
    print(f"[warn] {len(unused_wav)} .wav files without a matching video:", unused_wav[:5])


In [ ]:
# Repo dependencies from requirement.txt that this notebook needs.
# (transformers pinned to >=4.38 because 4.30.2 cannot be built on Python 3.12 - see notebook 01)
%pip install -q librosa "transformers>=4.38,<4.42"

# huggingface_hub is only needed for the one-time model download
%pip install -q "huggingface_hub>=0.19.3,<0.26"


In [ ]:

# Download wav2vec2-base-960h (same files app.py downloads from cbsjtu01/IMTalker)
NEEDED = ["config.json", "pytorch_model.bin", "preprocessor_config.json", "feature_extractor_config.json"]

if all((WAV2VEC_DIR / f).exists() for f in NEEDED):
    print("[skip] wav2vec2-base-960h already present")
else:
    from huggingface_hub import hf_hub_download
    WAV2VEC_DIR.mkdir(parents=True, exist_ok=True)
    ok = True
    for f in NEEDED:
        try:
            hf_hub_download(repo_id="cbsjtu01/IMTalker", filename=f"wav2vec2-base-960h/{f}", local_dir=str(CKPT_DIR))
        except Exception as e:
            print(f"[warn] could not get {f} from cbsjtu01/IMTalker ({e}); falling back to facebook/wav2vec2-base-960h")
            ok = False
            break
    if not ok:
        for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
            hf_hub_download(repo_id="facebook/wav2vec2-base-960h", filename=f, local_dir=str(WAV2VEC_DIR))
print("wav2vec2 model ready at", WAV2VEC_DIR)


In [ ]:

import sys
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import cv2
import librosa
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import Wav2Vec2FeatureExtractor

# The repo's own wav2vec2 wrapper (generator/wav2vec2.py) - used by FM.py AudioEncoder
from generator.wav2vec2 import Wav2VecModel

wav2vec2 = Wav2VecModel.from_pretrained(str(WAV2VEC_DIR), local_files_only=True)
wav2vec2.to(DEVICE).eval()
processor = Wav2Vec2FeatureExtractor.from_pretrained(str(WAV2VEC_DIR), local_files_only=True)
print("wav2vec2 loaded | hidden size:", wav2vec2.config.hidden_size)

FPS = 25                # generator/options/base_options.py --fps
SAMPLING_RATE = 16000   # --sampling_rate


In [ ]:

def count_frames(video_path: Path) -> int:
    # exact decoded frame count (CAP_PROP_FRAME_COUNT can be off by a few frames)
    cap = cv2.VideoCapture(str(video_path))
    n = 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        n += 1
    cap.release()
    return n

# Prefer the frame count stored in motion/*.pt (ground truth from notebook 01)
motion_dir = DATASET_DIR / "motion"
def get_T(stem: str, video_path: Path) -> int:
    m = motion_dir / f"{stem}.pt"
    if m.exists():
        return torch.load(str(m), map_location="cpu").shape[0]
    return count_frames(video_path)


In [ ]:

def extract_audio_features(wav_path: Path, video_path: Path, out_path: Path):
    T = get_T(video_path.stem, video_path)

    # 1) load + feature-processor (app.py process_audio / generate.py default_aud_loader)
    speech_array, sr = librosa.load(str(wav_path), sr=SAMPLING_RATE)
    inputs = processor(speech_array, sampling_rate=SAMPLING_RATE, return_tensors="pt").input_values[0]

    # 2) align waveform length to T video frames (FM.py AudioEncoder._pad_audio semantics)
    target_samples = int(T * SAMPLING_RATE / FPS)
    if inputs.shape[0] < target_samples:
        diff = target_samples - inputs.shape[0]
        inputs = F.pad(inputs.unsqueeze(0), (0, diff), mode="replicate").squeeze(0)
    else:
        inputs = inputs[:target_samples]

    # 3) wav2vec2 with per-frame interpolation (generator/wav2vec2.py linear_interpolation)
    with torch.no_grad():
        out = wav2vec2(inputs.unsqueeze(0).to(DEVICE), seq_len=T, output_hidden_states=False)
        feat = out.last_hidden_state.squeeze(0).float().cpu().numpy()   # (T, 768)
    assert feat.shape == (T, wav2vec2.config.hidden_size), f"unexpected feature shape {feat.shape}"
    np.save(str(out_path), feat.astype(np.float32))
    return T

audio_failures = []
todo = [v for v in cropped_videos
        if v.stem in wavs and (OVERWRITE or not (AUDIO_DIR / (v.stem + ".npy")).exists())]
print(f"Extracting audio features for {len(todo)} videos ...")
for v in tqdm(todo, desc="audio"):
    try:
        extract_audio_features(wavs[v.stem], v, AUDIO_DIR / (v.stem + ".npy"))
    except Exception as e:
        print(f"[FAIL] {v.name}: {e}")
        audio_failures.append(v.name)

print(f"audio/: {len(list(AUDIO_DIR.glob('*.npy')))} files (failures: {len(audio_failures)})")


In [ ]:

print("=" * 70)
print("AUDIO PREPROCESSING SUMMARY")
print("=" * 70)
print(f"audio/*.npy : {len(list(AUDIO_DIR.glob('*.npy')))}")
print(f"failures    : {len(audio_failures)}")
print()
print("Next: run 03_verify_dataset.ipynb, then train:")
print(f"  python generator/train.py --dataset_pat \"{DATASET_DIR}\" --exp_name generator_exp --batch_size 16 --iter 5000000 --lr 1e-4")
